# Project 30 — RAG Evaluation Dashboard

## Compare Chunking, Retrieval, and Groundedness

**Goal:** Systematic evaluation framework varying chunk sizes, measuring
accuracy and groundedness, producing a comparison dashboard.

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

### What You'll Learn
1. Evaluation set design with ground-truth answers
2. Chunking strategy comparison
3. Answer accuracy and groundedness measurement
4. Dashboard-style reporting

### Prerequisites
- **Ollama** with `nomic-embed-text` and `qwen3:8b`

In [ ]:
!pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    print(f"Ollama running — {len(r.json().get('models',[]))} model(s)")
except Exception as e:
    print(f"Cannot reach Ollama: {e}")

## Step 2 — Source Document and Evaluation Set

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.0)
emb = OllamaEmbeddings(model="nomic-embed-text")

source = """Kubernetes orchestrates containers across clusters. Key components:
pods (smallest deployable unit), services (stable networking), deployments
(declarative updates), ingress (external HTTP). The control plane runs the
API server, scheduler, controller manager, and etcd (key-value store).
Worker nodes run kubelet and container runtime (containerd). HPA scales
pods based on CPU/memory metrics. Rolling updates enable zero-downtime
deployments. Namespaces provide logical isolation. ConfigMaps store
configuration, Secrets store credentials. PersistentVolumes abstract
storage. StatefulSets manage stateful applications."""

evals = [
    {"q": "What is the smallest unit in Kubernetes?", "a": "pods"},
    {"q": "What provides stable networking?", "a": "services"},
    {"q": "What does HPA scale based on?", "a": "CPU"},
    {"q": "How does K8s handle zero-downtime updates?", "a": "rolling"},
    {"q": "What manages configuration data?", "a": "ConfigMaps"},
    {"q": "What stores sensitive information?", "a": "Secrets"},
    {"q": "What is etcd?", "a": "key-value"},
    {"q": "What manages stateful applications?", "a": "StatefulSets"},
]
print(f"Source: {len(source)} chars, Eval set: {len(evals)} QA pairs")

## Step 3 — Compare Chunking Strategies

In [ ]:
configs = [
    ("tiny", 80, 10), ("small", 150, 20),
    ("medium", 300, 40), ("large", 500, 60),
]

doc = Document(page_content=source, metadata={"source": "k8s_doc"})
all_results = {}

qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="Answer from context only. Be concise.\nContext: {context}\nQ: {question}\nA:")

for name, size, overlap in configs:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_documents([doc])
    vs = Chroma.from_documents(chunks, emb, collection_name=f"eval_{name}")
    qa = RetrievalQA.from_chain_type(
        llm=llm, chain_type="stuff",
        retriever=vs.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True,
        chain_type_kwargs={"prompt": qa_prompt})

    cr = []
    for item in evals:
        r = qa.invoke({"query": item["q"]})
        correct = item["a"].lower() in r["result"].lower()
        cr.append({"q": item["q"], "exp": item["a"],
                   "got": r["result"][:80], "ok": correct})

    acc = sum(r["ok"] for r in cr) / len(cr)
    all_results[name] = {"acc": acc, "n": len(chunks), "det": cr, "sz": size}
    print(f"  {name:>6}: {len(chunks)} chunks, accuracy={acc:.0%}")

## Step 4 — Dashboard Summary

In [ ]:
print("\n" + "="*55)
print("RAG EVALUATION DASHBOARD")
print("="*55)
print(f"\n{'Strategy':<10} {'Size':>5} {'Chunks':>7} {'Accuracy':>9}")
print("-" * 35)
for n, d in all_results.items():
    status = "GOOD" if d["acc"] >= 0.8 else "CHECK" if d["acc"] >= 0.6 else "POOR"
    print(f"{n:<10} {d['sz']:>5} {d['n']:>7} {d['acc']:>8.0%}  {status}")

best = max(all_results, key=lambda k: all_results[k]["acc"])
print(f"\nBest strategy: {best} ({all_results[best]['acc']:.0%} accuracy)")

## Step 5 — Per-Question Breakdown

In [ ]:
b = all_results[best]
print(f"Details for '{best}' strategy:\n")
for r in b["det"]:
    icon = "✓" if r["ok"] else "✗"
    print(f"  {icon} {r['q']}")
    print(f"    Expected: {r['exp']} | Got: {r['got'][:60]}")
    print()

## Step 6 — Groundedness Evaluation

In [ ]:
jp = ChatPromptTemplate.from_messages([
    ("system", "Is this answer grounded in the source? "
     "Return GROUNDED or HALLUCINATED with brief explanation."),
    ("human", "Source: {source}\nAnswer: {answer}\nVerdict:")
])
jc = jp | llm | StrOutputParser()

print("Groundedness Evaluation:\n")
gc = 0
for r in b["det"]:
    v = jc.invoke({"source": source, "answer": r["got"]})
    g = "grounded" in v.lower().split("\n")[0].lower()
    if g: gc += 1
    print(f"  {'✓' if g else '✗'} {r['q'][:45]}")
    print(f"    {v.strip()[:80]}")
    print()

print(f"Groundedness: {gc}/{len(b['det'])} ({gc/len(b['det']):.0%})")

## Step 7 — Final Report

In [ ]:
print("="*55)
print("FINAL RAG EVALUATION REPORT")
print("="*55)
print(f"\nBest chunking: {best} ({all_results[best]['sz']} chars)")
print(f"Chunks: {all_results[best]['n']}")
print(f"Answer accuracy: {all_results[best]['acc']:.0%}")
print(f"Groundedness: {gc}/{len(b['det'])} ({gc/len(b['det']):.0%})")
print(f"\nRecommendation: ", end="")
if all_results[best]["acc"] >= 0.8 and gc/len(b['det']) >= 0.8:
    print("This configuration is production-ready.")
elif all_results[best]["acc"] >= 0.6:
    print("Acceptable but could be improved.")
else:
    print("Needs significant improvement.")

## Limitations

1. **Simple accuracy:** Contains-check is crude; needs semantic scoring.
2. **Single source:** Real evaluations need diverse documents.
3. **No retrieval metrics:** Only end-to-end accuracy, not nDCG or MRR.

## What You Learned

| Concept | What We Did |
|---|---|
| **Evaluation set** | Ground-truth QA pairs |
| **Chunking comparison** | 4 chunk sizes |
| **Accuracy measurement** | Contains-based check |
| **Groundedness** | LLM judge for hallucination |
| **Dashboard** | Summary with best configuration |